In [7]:
import pandas as pd

import plotly.graph_objects as go
import numpy as np

# 1. Lấy danh sách các quốc gia duy nhất (sắp xếp theo Alphabet)
countries = ['VNM', 'THA', 'PHL', 'SGP', 'CHN', 'IND', 'IRL', 'DEU', 'ZAF', 'USA', 'JPN']
countries.sort()

file_path = 'cleaned_data/information/panel_macro_cleaned.csv'
try:
    df = pd.read_csv(file_path)
except FileNotFoundError:
    print(f"Không tìm thấy file tại {file_path}.")
    
fig = go.Figure()

# 2. Vòng lặp: Thêm toàn bộ các traces (đường/cột) cho TẤT CẢ quốc gia vào biểu đồ
for country in countries:
    country_df = df[df['Country'] == country].sort_values('Year')
    
    # Trace 1: Cột Cán cân thương mại
    fig.add_trace(go.Bar(
        x=country_df['Year'],
        y=country_df['Trade_Balance_USD'],
        name=f'Cán cân thương mại',
        marker_color=np.where(country_df['Trade_Balance_USD'] < 0, 'red', 'green'),
        visible=(country == 'VNM') # Mặc định chỉ hiển thị VNM ban đầu
    ))

    # Trace 2: Đường Độ mở kinh tế
    fig.add_trace(go.Scatter(
        x=country_df['Year'],
        y=country_df['Economic_Openness_Pct'],
        name=f'Độ mở kinh tế',
        yaxis='y2',
        line=dict(color='blue', width=3),
        visible=(country == 'VNM') # Mặc định chỉ hiển thị VNM ban đầu
    ))

# 3. Xây dựng logic cho Menu Dropdown
buttons = []
for i, country in enumerate(countries):
    # Mỗi quốc gia có 2 traces (Bar và Scatter), tạo mảng boolean để bật đúng 2 traces đó
    visibility = [False] * (len(countries) * 2)
    visibility[i*2] = True      # Bật Bar chart của quốc gia thứ i
    visibility[i*2 + 1] = True  # Bật Scatter chart của quốc gia thứ i
    
    button = dict(
        label=country,
        method="update",
        args=[
            {"visible": visibility}, # Hành động 1: Cập nhật dữ liệu hiển thị
            {"title": f"{country}: Chuyển đổi mô hình Thương mại (1990 - 2024)"} # Hành động 2: Đổi tiêu đề
        ]
    )
    buttons.append(button)

# 4. Cập nhật Layout với Dropdown và Trục Y kép
# Đặt chỉ mục mặc định của Dropdown vào 'VNM'
default_index = countries.index('VNM') if 'VNM' in countries else 0

fig.update_layout(
    updatemenus=[
        dict(
            active=default_index,
            buttons=buttons,
            x=1.1, # Đẩy menu ra ngoài góc phải
            y=1.15,
            xanchor="right",
            yanchor="top"
        )
    ],
    title='VNM: Chuyển đổi mô hình Thương mại (1990 - 2024)',
    yaxis=dict(title='Cán cân thương mại (USD)'),
    yaxis2=dict(title='Độ mở kinh tế (% GDP)', overlaying='y', side='right'),
    barmode='group',
    height=600,
    margin=dict(r=100) # Mở rộng lề phải để không bị cắt chữ ở trục Y2
)

fig.show()
fig.write_html(f"fig/Mô hình thương mại.html")

In [11]:
# Bieu do tuong quan moi chi so cho tung quoc gia
selected_countries = ['VNM', 'THA', 'PHL', 'SGP', 'CHN', 'IND', 'IRL', 'DEU', 'ZAF', 'USA', 'JPN']
df_filtered = df[df["Country"].isin(selected_countries)].copy()

numeric_cols = df_filtered.select_dtypes(include=[np.number]).columns.tolist()
exclude_cols = {"Year"}
corr_cols = [col for col in numeric_cols if col not in exclude_cols]

for country, g in df_filtered.groupby("Country"):
    corr_matrix = g[corr_cols].corr()
    #print(f"Correlation matrix for {country}:")
    #display(corr_matrix)

    heatmap = go.Heatmap(
        z=corr_matrix.values,
        x=corr_matrix.columns,
        y=corr_matrix.index,
        colorscale="RdBu",
        zmin=-1,
        zmax=1,
        colorbar=dict(title="Correlation")
    )

    fig_corr = go.Figure(data=heatmap)
    fig_corr.update_layout(
        title=f"Tuong quan cac chi so - {country}",
        height=900,
        margin=dict(l=140, r=50, t=80, b=140)
    )
    fig_corr.write_html(f"fig/correlation_heatmap_{country}.html")

Nhìn chung, EO của hầu hết các quốc gia đều có dấu hiện sụt giảm sau khủng hoảng tài chính năm 2008 và có dấu hiệu phục hồi dần dần.

1. Nhóm Chuyển đổi từ Nhập siêu sang Xuất siêu (VNM, THA)
Các quốc gia này đã thành công trong việc thay đổi bản chất nền kinh tế từ tiêu thụ sang sản xuất xuất khẩu.

Việt Nam (VNM):

Cột mốc: Trước 2012 chủ yếu là nhập siêu.

Khủng hoảng: Chịu ảnh hưởng bởi khủng hoảng 2008 khiến độ mở kinh tế sụt giảm tạm thời.

Hiện tại: Trở thành "ngôi sao" xuất siêu với độ mở kinh tế cực cao (vượt 180% GDP).

Thái Lan (THA):

Cột mốc: Khủng hoảng tài chính Á châu 1997 là bước ngoặt buộc THA chuyển từ nhập siêu sang xuất siêu để phục hồi.

Hiện tại: Độ mở kinh tế cao (~130% GDP) nhưng cán cân thương mại gần đây biến động mạnh, có dấu hiệu nhập siêu trở lại vào năm 2022.

2. Nhóm Xuất siêu Bền vững (DEU, IRL, SGP)
Những quốc gia này duy trì thặng dư thương mại khổng lồ, đóng vai trò là nguồn cung hàng hóa và dịch vụ cho toàn cầu.

Đức (DEU): Duy trì xuất siêu ổn định ở mức cực cao (>200 tỷ USD). Độ mở kinh tế tăng dần cho thấy sự phụ thuộc vào thị trường bên ngoài.

Ireland (IRL): Mô hình tăng trưởng bùng nổ sau 2015. Độ mở kinh tế "khủng" (>240% GDP) biến đây thành trung tâm trung chuyển xuất khẩu của các tập đoàn đa quốc gia.

Singapore (SGP): Xuất siêu bền vững gắn liền với vị thế cảng biển và trung tâm tài chính. Độ mở kinh tế luôn ở mức cao nhất thế giới (>300% GDP). Sau năm 2008, EO của SGP có sự chững lại, không cao bằng những năm đầu 2000.

3. Nhóm "Thị trường Tiêu thụ" (USA, IND, PHL)
Đây là những nền kinh tế lấy nội lực tiêu dùng làm trọng tâm, chấp nhận nhập siêu để phục vụ nhu cầu trong nước.

Hoa Kỳ (USA): Nhập siêu ngày càng sâu, chạm ngưỡng 1.000 tỷ USD. Độ mở kinh tế thấp (~25% GDP) cho thấy sức mạnh nội địa khổng lồ.

Ấn Độ (IND): Nhập siêu triền miên. Khủng hoảng 2008 và 2012 (khủng hoảng nợ công châu Âu) làm chậm đà tăng độ mở kinh tế của quốc gia này.

Philippines (PHL): (Dựa trên xu hướng khu vực và dữ liệu tương đồng) Thường xuyên nhập siêu do phụ thuộc vào hàng hóa nhập khẩu và kiều hối hỗ trợ tiêu dùng.

4. Nhóm Đảo chiều và Biến động (JPN, CHN, ZAF)
Nhật Bản (JPN): Bước ngoặt 2011 (thảm họa kép động đất - sóng thần) đã chấm dứt kỷ nguyên xuất siêu huy hoàng, đẩy Nhật vào tình trạng nhập siêu kéo dài do chi phí năng lượng.

Trung Quốc (CHN): (Dựa trên bối cảnh chung) Là quốc gia xuất siêu lớn nhất thế giới, tuy nhiên đang có xu hướng giảm dần độ mở kinh tế để tập trung vào "Tuần hoàn nội bộ".

Nam Phi (ZAF): (Dựa trên bối cảnh chung) Cán cân thương mại phụ thuộc nặng nề vào giá tài nguyên/khoáng sản thế giới, thường xuyên biến động theo chu kỳ hàng hóa.

In [12]:
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import ipywidgets as widgets
from IPython.display import display, clear_output

# 1. ĐỌC DỮ LIỆU
file_path = 'cleaned_data/information/panel_macro_cleaned.csv'
try:
    df = pd.read_csv(file_path)
except FileNotFoundError:
    print(f"Không tìm thấy file tại {file_path}.")

# Lấy danh sách quốc gia đối chiếu
available_countries = countries = ['THA', 'PHL', 'SGP', 'CHN', 'IND', 'IRL', 'DEU', 'ZAF', 'USA', 'JPN']
available_countries.sort()

# 2. ĐỊNH NGHĨA CÁC CHỈ SỐ CẦN VẼ (Cập nhật 8 biểu đồ)
indicators = [
    {'col': 'Economic_Openness_Pct', 'title': '1. Độ mở Kinh tế (% GDP)', 'ylabel': 'Tỷ lệ (%)'},
    {'col': 'GDP_Current_USD', 'title': '2. Quy mô GDP (Tỷ USD)', 'ylabel': 'Tỷ USD', 'divide_by': 1e9},
    {'col': 'GDP_Growth_Pct', 'title': '3. Tăng trưởng GDP (%)', 'ylabel': 'Tốc độ (%)'},
    {'col': 'GDP_GNI_Gap_Pct', 'title': '4. GNI & GDP Gap (% GDP)', 'ylabel': 'Gap (%)'},
    {'col': 'Inflation_CPI_Pct', 'title': '5. Lạm phát (CPI %)', 'ylabel': 'Lạm phát (%)'},
    {'col': 'FDI_to_GDP_Pct', 'title': '6. Tỷ trọng FDI mới / GDP (%)', 'ylabel': 'FDI / GDP (%)'},
    {'col': 'Lending_Interest_Rate_Pct', 'title': '7. Lãi suất cho vay (%)', 'ylabel': 'Lãi suất (%)'},
    {'col': 'Remittances_Pct_GDP', 'title': '8. Kiều hối / GDP (%)', 'ylabel': 'Tỷ lệ (%)'}
]

# 3. HÀM VẼ DASHBOARD TƯƠNG TÁC
def update_plotly_dashboard(selected_country):
    df_plot = df[df['Country'].isin(['VNM', selected_country])].copy()
    
    # Tạo khung 8 biểu đồ (4 hàng x 2 cột)
    fig = make_subplots(
        rows=4, cols=2, 
        subplot_titles=[ind['title'] for ind in indicators],
        vertical_spacing=0.08, horizontal_spacing=0.1
    )
    
    # Tọa độ 8 biểu đồ trên lưới
    coords = [(1,1), (1,2), (2,1), (2,2), (3,1), (3,2), (4,1), (4,2)]
    
    colors = {'VNM': '#d62728', selected_country: '#1f77b4'}
    
    for i, ind in enumerate(indicators):
        row, col_idx = coords[i]
        col_name = ind['col']
        
        plot_data = df_plot.copy()
        if 'divide_by' in ind:
            plot_data[col_name] = plot_data[col_name] / ind['divide_by']
            
        # Vẽ đường cho từng quốc gia
        for country in ['VNM', selected_country]:
            country_data = plot_data[plot_data['Country'] == country]
            
            # Cài đặt độ nét: VNM nét liền đậm, quốc gia kia nét thường
            line_width = 3.5 if country == 'VNM' else 2.0
            
            fig.add_trace(
                go.Scatter(
                    x=country_data['Year'], 
                    y=country_data[col_name],
                    mode='lines+markers',
                    name=country,
                    line=dict(color=colors[country], width=line_width),
                    hovertemplate=f"<b>{country}</b><br>Năm: %{{x}}<br>Giá trị: %{{y:,.2f}}<extra></extra>",
                    showlegend=(i==0) # Chỉ hiển thị chú thích 1 lần
                ),
                row=row, col=col_idx
            )
            
        # Thêm đường mốc 0 (Baseline)
        fig.add_hline(y=0, line_dash="dash", line_color="black", opacity=0.5, row=row, col=col_idx)
        
        # Cập nhật trục X và Y
        fig.update_yaxes(title_text=ind['ylabel'], tickformat=",", row=row, col=col_idx)
        fig.update_xaxes(title_text="Năm", range=[1990, 2024], dtick=5, row=row, col=col_idx)

    # Cấu hình giao diện tổng thể
    fig.update_layout(
        height=1800, 
        # width=None, # Để None để tự dãn theo khung Notebook
        autosize=True, # Tự động điều chỉnh kích thước
        
        # TỐI ƯU KHOẢNG TRẮNG: Giảm bớt lề (margin)
        margin=dict(l=50, r=20, t=100, b=50), 
        
        title_text=f"SO SÁNH KINH TẾ VĨ MÔ: VIỆT NAM VÀ {selected_country}",
        title_font=dict(size=24, color='black'),
        title_x=0.05, # Căn tiêu đề lệch trái một chút để cân bằng
        
        hovermode="x unified",
        template="plotly_white",
        
        # Đưa Legend lên trên và dàn ngang để không chiếm diện tích bên phải
        legend=dict(
            orientation="h",
            yanchor="bottom",
            y=1.02,
            xanchor="right",
            x=1
        )
    )
    
    # ÉP BIỂU ĐỒ DÃN HẾT CHIỀU RỘNG TRÌNH DUYỆT
    fig.show(config={'responsive': True})
    fig.write_html(f"fig/So_sanh_VNM_va_{selected_country}.html")
    
# 4. TẠO MENU THẢ XUỐNG
dropdown = widgets.Dropdown(
    options=available_countries,
    value='THA', 
    description='Chọn Quốc gia:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='300px')
)

# Gắn widget với hàm cập nhật
out = widgets.interactive_output(update_plotly_dashboard, {'selected_country': dropdown})

# Hiển thị
display(dropdown, out)


Dropdown(description='Chọn Quốc gia:', index=7, layout=Layout(width='300px'), options=('CHN', 'DEU', 'IND', 'I…

Output()